# 한우 비문 개체식별 — Colab 학습 노트북

**목표**: Zenodo Beef Cattle Muzzle DB(268두 / 4,923장)로 **임베딩 추출기**를 학습한다.
분류기가 아니다. 새 개체를 등록만 하면 재학습 없이 인식되는 구조를 만든다.

**중요 — 실행 전에 GPU 켜라**: 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **T4 GPU**

**예상 소요**: 다운로드 3~5분 + 학습 15~25분 (무료 T4 기준)

**출력물**:
1. `muzzle_encoder.pt` — 임베딩 백본
2. `muzzle_encoder.onnx` — 배포용
3. `openset_results.csv` — 임계값별 오배정률/보류율 표 (기획서 §9 지표에 그대로 들어감)

## 0. 환경 확인

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU가 꺼져 있다. 런타임 → 런타임 유형 변경 → T4 GPU"

In [ ]:
!pip -q install timm==1.0.9
import timm; print("timm:", timm.__version__)

## 1. 데이터 다운로드

Zenodo 6324361 / 643.7MB / CC BY 4.0.
**반드시 `/content`(로컬 SSD)에 푼다. 구글 드라이브에 풀면 작은 파일 5천 개 I/O로 학습이 10배 느려진다.**

In [ ]:
import os, hashlib, zipfile, glob, shutil

ZIP  = "/content/muzzle.zip"
ROOT = "/content/muzzle"
MD5  = "f2b19d9bab2fc668460a796259944204"

# 이미 받아둔 zip이 있으면 재사용한다 (643MB 다시 받지 않는다)
if not os.path.exists(ZIP):
    found = []
    for d in ["/content", "/", "/root", "/content/drive/MyDrive", "/content/sample_data"]:
        found += glob.glob(os.path.join(d, "BeefCattle_Muzzle_database.zip"))
    if found:
        print("기존 파일 발견 →", found[0])
        shutil.move(found[0], ZIP)
    else:
        print("다운로드 시작 (3~5분)")
        !wget -q --show-progress -O {ZIP} "https://zenodo.org/records/6324361/files/BeefCattle_Muzzle_database.zip?download=1"

print("파일 크기:", round(os.path.getsize(ZIP)/1e6, 1), "MB (기대: 643.7 MB)")

# 무결성 확인 (공식 md5) — 청크로 읽는다
_h = hashlib.md5()
with open(ZIP, "rb") as f:
    for b in iter(lambda: f.read(1 << 22), b""):
        _h.update(b)
h = _h.hexdigest()
print("md5:", h, "|", "OK" if h == MD5 else "★불일치 → 파일이 깨졌다. ZIP 지우고 이 셀 재실행")

if not os.path.isdir(ROOT):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(ROOT)
print("압축 해제 완료")
!du -sh {ROOT}

In [ ]:
# 폴더 구조 파악 — 개체 1마리 = 폴더 1개
import glob, pandas as pd

exts = ("*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG")
paths = []
for e in exts:
    paths += glob.glob(os.path.join(ROOT, "**", e), recursive=True)
print("총 이미지:", len(paths))

df = pd.DataFrame({"path": paths})
df["cow"] = df.path.apply(lambda p: os.path.basename(os.path.dirname(p)))
print("총 개체 수:", df.cow.nunique())
print(df.groupby("cow").size().describe())
df.head()

In [ ]:
# 샘플 눈으로 확인 — 이 단계 건너뛰지 마라. 폴더 구조가 다르면 여기서 바로 드러난다.
import matplotlib.pyplot as plt, cv2
fig, ax = plt.subplots(2, 5, figsize=(15,6))
for a, p in zip(ax.ravel(), df.sample(10, random_state=0).path):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    a.imshow(img); a.set_title(f"{os.path.basename(os.path.dirname(p))} {img.shape[:2]}", fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

## 2. 분할 — 여기가 이 노트북의 핵심이다

개체 268마리를 **마리 단위로** 나눈다.

- **학습 200마리** → 임베딩 학습용
- **평가 68마리** → 학습 중 한 번도 본 적 없는 개체. 이걸로 "새로 등록한 소를 알아보는가"를 측정한다.

이미지 단위로 랜덤 분할하면 같은 소가 학습·평가에 동시에 들어가서 정확도가 부풀려진다.
논문·발표에서 가장 먼저 찔리는 지점이다.

In [ ]:
import numpy as np
SEED = 42
rng = np.random.RandomState(SEED)

cows = sorted(df.cow.unique())
rng.shuffle(cows)
n_unseen = 68
unseen_cows = set(cows[:n_unseen])
train_cows  = [c for c in cows if c not in unseen_cows]

df_seen   = df[~df.cow.isin(unseen_cows)].copy()
df_unseen = df[ df.cow.isin(unseen_cows)].copy()

# 학습 개체 내부에서 다시 train/val (같은 소의 다른 사진)
df_seen["r"] = rng.rand(len(df_seen))
df_tr = df_seen[df_seen.r >= 0.12].copy()
df_va = df_seen[df_seen.r <  0.12].copy()

cow2lab = {c:i for i,c in enumerate(sorted(df_seen.cow.unique()))}
df_tr["label"] = df_tr.cow.map(cow2lab)
df_va["label"] = df_va.cow.map(cow2lab)
NUM_CLASSES = len(cow2lab)

print(f"학습 개체 {len(train_cows)}두 / 이미지 train {len(df_tr)}, val {len(df_va)}")
print(f"미학습 개체 {len(unseen_cows)}두 / 이미지 {len(df_unseen)}")

## 3. 설정

`GRAY = True`가 그레이스케일 실험 스위치다.

**왜 그레이스케일인가**: 비문의 정보는 코 표면의 **융기 무늬(texture)** 이지 색이 아니다.
이 데이터셋은 Angus(검은 코) + Angus×Hereford(밝은/분홍 코) 혼합이라, 컬러로 학습하면
모델이 무늬 대신 **코 피부색**을 지름길로 배울 수 있다. 그 지름길은 한우로 넘어가는 순간 무너진다.

**그러니 고르지 말고 둘 다 돌려라.** 아래 CONFIG만 바꿔서 2회 학습 → 결과 비교 → 표로 제출.
한 번에 20분이라 비용이 없다.

In [ ]:
CONFIG = dict(
    exp_name  = "gray_clahe",     # 실험 이름 (결과 CSV에 기록됨)
    GRAY      = True,             # ★ 실험 스위치: True / False
    CLAHE     = True,             # 그레이스케일일 때 대비 강화 (GRAY=False면 무시)
    model     = "efficientnet_b0",# 빠름. 선행연구 정합 원하면 "tf_efficientnetv2_s.in1k"
    img_size  = 224,
    embed_dim = 512,
    epochs    = 25,
    batch     = 64,
    lr        = 3e-4,
    arc_s     = 30.0,
    arc_m     = 0.30,
    lowq_aug  = True,             # 저화질 시뮬 증강 (스마트폰/저해상도 도메인 갭 대비)
)
for k,v in CONFIG.items(): print(f"{k:10s} = {v}")

## 4. 데이터셋 & 증강

증강에서 두 가지가 중요하다.

1. **저화질 시뮬레이션** — 학습 데이터는 미러리스 + 70-300mm 망원으로 찍은 고화질이다.
   실제 앱은 스마트폰이다. 축소→확대 + JPEG 재압축을 랜덤으로 걸어서 미리 무너뜨려 둔다.
   이게 없으면 Zenodo 정확도만 높고 현장에서 죽는다.
2. **좌우 반전 금지** — 비문은 좌우 비대칭 패턴이다. flip을 넣으면 없는 개체를 만들어낸다.

In [ ]:
import cv2, random
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

def low_quality(img_np, p=0.5):
    # 축소-확대 + JPEG 재압축으로 저화질 시뮬레이션
    if random.random() > p:
        return img_np
    h, w = img_np.shape[:2]
    f = random.uniform(0.25, 0.7)
    small = cv2.resize(img_np, (max(8,int(w*f)), max(8,int(h*f))), interpolation=cv2.INTER_AREA)
    img_np = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    q = random.randint(30, 80)
    ok, enc = cv2.imencode(".jpg", img_np[:, :, ::-1], [int(cv2.IMWRITE_JPEG_QUALITY), q])
    if ok:
        img_np = cv2.imdecode(enc, cv2.IMREAD_COLOR)[:, :, ::-1]
    return np.ascontiguousarray(img_np)

MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def build_tf(train, size):
    if train:
        return T.Compose([
            T.RandomResizedCrop(size, scale=(0.65,1.0), ratio=(0.85,1.18)),
            T.RandomRotation(15, fill=0),
            T.RandomApply([T.GaussianBlur(3, (0.1,1.5))], p=0.3),
            T.ToTensor(), T.Normalize(MEAN, STD),
            T.RandomErasing(p=0.25, scale=(0.02,0.12)),
        ])
    return T.Compose([T.Resize(int(size*1.14)), T.CenterCrop(size),
                      T.ToTensor(), T.Normalize(MEAN, STD)])

class MuzzleDS(Dataset):
    def __init__(self, df, train, cfg, with_label=True):
        self.paths = df.path.tolist()
        self.labels = df.label.tolist() if with_label else [0]*len(df)
        self.train, self.cfg = train, cfg
        self.tf = build_tf(train, cfg["img_size"])
        # 컬러 모드일 때만 색 증강 (색 지름길 억제)
        self.color_jit = T.Compose([
            T.ColorJitter(0.35, 0.35, 0.35, 0.06),
            T.RandomGrayscale(p=0.3),
        ]) if (train and not cfg["GRAY"]) else None

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        img = cv2.imread(self.paths[i])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.train and self.cfg["lowq_aug"]:
            img = low_quality(img)
        if self.cfg["GRAY"]:
            g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            if self.cfg["CLAHE"]:
                g = _clahe.apply(g)
            img = np.stack([g,g,g], -1)          # ImageNet 사전학습이 3채널을 요구하므로 복제
        pil = Image.fromarray(img)
        if self.color_jit is not None:
            pil = self.color_jit(pil)
        return self.tf(pil), self.labels[i]

ds_tr = MuzzleDS(df_tr, True,  CONFIG)
ds_va = MuzzleDS(df_va, False, CONFIG)
dl_tr = DataLoader(ds_tr, batch_size=CONFIG["batch"], shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
dl_va = DataLoader(ds_va, batch_size=CONFIG["batch"], shuffle=False, num_workers=2, pin_memory=True)
print(len(ds_tr), len(ds_va))

In [ ]:
# 증강 결과 눈으로 확인 (모델이 실제로 보는 그림)
inv = lambda t: (t.permute(1,2,0).numpy()*np.array(STD)+np.array(MEAN)).clip(0,1)
fig, ax = plt.subplots(1,6, figsize=(16,3))
for a in ax:
    x,_ = ds_tr[random.randrange(len(ds_tr))]
    a.imshow(inv(x)); a.axis("off")
plt.suptitle(f"GRAY={CONFIG['GRAY']} CLAHE={CONFIG['CLAHE']}"); plt.show()

## 5. 모델 — 백본 + ArcFace

ArcFace를 쓰는 이유: 일반 softmax는 "이 268마리 중 누구냐"만 잘 맞추면 되지만,
ArcFace는 **같은 개체는 서로 가깝게, 다른 개체는 멀게** 임베딩 공간 자체를 정렬한다.
학습에 없던 새 개체에도 그 성질이 유지된다. 등록 기반 인식에는 이게 맞다.

In [ ]:
import torch.nn as nn, torch.nn.functional as F, timm

class ArcFace(nn.Module):
    def __init__(self, in_dim, n_cls, s=30.0, m=0.30):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_cls, in_dim)); nn.init.xavier_normal_(self.W)
        self.s, self.m = s, m
    def forward(self, x, y):
        cos = F.linear(F.normalize(x), F.normalize(self.W)).clamp(-1+1e-7, 1-1e-7)
        th  = torch.acos(cos)
        oh  = F.one_hot(y, cos.size(1)).float()
        out = torch.cos(th + self.m*oh)
        return out * self.s

class Encoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(cfg["model"], pretrained=True, num_classes=0)
        d = self.backbone.num_features
        self.neck = nn.Sequential(nn.BatchNorm1d(d), nn.Dropout(0.2),
                                  nn.Linear(d, cfg["embed_dim"]), nn.BatchNorm1d(cfg["embed_dim"]))
    def forward(self, x):
        return self.neck(self.backbone(x))   # 정규화는 사용하는 쪽에서

dev = "cuda"
enc  = Encoder(CONFIG).to(dev)
head = ArcFace(CONFIG["embed_dim"], NUM_CLASSES, CONFIG["arc_s"], CONFIG["arc_m"]).to(dev)
print("파라미터:", sum(p.numel() for p in enc.parameters())/1e6, "M")

## 6. 학습

In [ ]:
import time, math
torch.manual_seed(SEED)

opt   = torch.optim.AdamW(list(enc.parameters())+list(head.parameters()), lr=CONFIG["lr"], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CONFIG["lr"], total_steps=CONFIG["epochs"]*len(dl_tr), pct_start=0.15)
scaler = torch.cuda.amp.GradScaler()
crit  = nn.CrossEntropyLoss(label_smoothing=0.1)

best = 0.0
for ep in range(1, CONFIG["epochs"]+1):
    enc.train(); head.train(); t0=time.time(); tot=0; run=0
    for x,y in dl_tr:
        x,y = x.to(dev,non_blocking=True), y.to(dev,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss = crit(head(enc(x), y), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()*x.size(0); tot += x.size(0)

    enc.eval(); head.eval(); c=n=0
    with torch.no_grad(), torch.cuda.amp.autocast():
        for x,y in dl_va:
            x,y = x.to(dev), y.to(dev)
            logit = F.linear(F.normalize(enc(x)), F.normalize(head.W))
            c += (logit.argmax(1)==y).sum().item(); n += y.numel()
    acc = c/n
    print(f"ep{ep:02d}  loss {run/tot:.3f}  val_acc {acc:.4f}  ({time.time()-t0:.0f}s)")
    if acc > best:
        best = acc
        torch.save({"enc": enc.state_dict(), "cfg": CONFIG}, "/content/muzzle_encoder.pt")
print("best val acc (학습에 본 개체 기준):", round(best,4))

## 7. ★ 개방집합 평가 — 발표·논문에 들어갈 숫자는 여기서 나온다

학습에 없던 **68마리**를 대상으로 실제 운영 절차를 그대로 재현한다.

1. 각 소마다 3장을 **등록(gallery)** — 앱에서 농가가 찍는 그 3장이다
2. 나머지 사진을 **조회(probe)** — 매일 카메라가 던지는 그 질의다
3. 코사인 유사도 최댓값이 임계값 미만이면 **"미확정" 보류** (기획서 §2 설계 그대로)

산출 지표: **오배정률**(ID를 붙였는데 틀린 비율), **보류율**, **커버리지 내 정확도**.
기획서 §9의 `오배정률 ≤ 0.02`를 검증하는 유일한 방법이다. 268-클래스 분류 정확도로는 이 숫자가 안 나온다.

In [ ]:
ckpt = torch.load("/content/muzzle_encoder.pt"); enc.load_state_dict(ckpt["enc"]); enc.eval()

@torch.no_grad()
def embed(paths, cfg, bs=64):
    tmp = pd.DataFrame({"path": paths}); tmp["label"]=0
    dl = DataLoader(MuzzleDS(tmp, False, cfg), batch_size=bs, num_workers=2)
    out=[]
    for x,_ in dl:
        with torch.cuda.amp.autocast():
            out.append(F.normalize(enc(x.to(dev))).float().cpu())
    return torch.cat(out)

# 개체별 3장 등록 / 나머지 조회
g_paths, g_ids, p_paths, p_ids = [],[],[],[]
for cow, grp in df_unseen.groupby("cow"):
    ps = grp.path.sample(frac=1.0, random_state=SEED).tolist()
    if len(ps) < 5: continue
    g_paths += ps[:3]; g_ids += [cow]*3
    p_paths += ps[3:]; p_ids += [cow]*(len(ps)-3)

G = embed(g_paths, CONFIG); P = embed(p_paths, CONFIG)
uniq = sorted(set(g_ids))
cent = torch.stack([F.normalize(G[[i for i,c in enumerate(g_ids) if c==u]].mean(0), dim=0) for u in uniq])

sim = P @ cent.T
best_sim, best_idx = sim.max(1)
pred = [uniq[i] for i in best_idx.tolist()]
truth = p_ids
print(f"등록 {len(uniq)}두 x 3장 / 조회 {len(p_paths)}장")
print("Top-1 (임계값 없음):", round(float(np.mean([a==b for a,b in zip(pred,truth)])),4))

In [ ]:
rows=[]
for t in np.arange(0.0, 0.96, 0.05):
    assigned = best_sim.numpy() >= t
    n = len(truth)
    correct = np.array([a==b for a,b in zip(pred,truth)])
    misassign = ((assigned) & (~correct)).sum() / n     # 오배정률 (전체 대비)
    hold      = (~assigned).sum() / n                    # 보류율
    cov_acc   = correct[assigned].mean() if assigned.sum() else float("nan")
    rows.append(dict(exp=CONFIG["exp_name"], gray=CONFIG["GRAY"], threshold=round(float(t),2),
                     misassign_rate=round(float(misassign),4), hold_rate=round(float(hold),4),
                     acc_within_coverage=round(float(cov_acc),4)))
res = pd.DataFrame(rows)

# 오배정률 2% 이하를 만족하는 최소 임계값
ok = res[res.misassign_rate <= 0.02]
print(res.to_string(index=False))
if len(ok):
    r = ok.iloc[0]
    print(f"\n★ 운영 임계값 권장: {r.threshold}  → 오배정률 {r.misassign_rate}, 보류율 {r.hold_rate}")
else:
    print("\n★ 오배정률 2% 달성 불가 — 학습 에폭/모델 키우거나 등록 장수를 3→5로 늘려라")

import os
res.to_csv("/content/openset_results.csv", mode="a", header=not os.path.exists("/content/openset_results.csv"), index=False)

In [ ]:
# 임계값-오배정률-보류율 곡선 (발표 슬라이드용 그림 1장)
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(res.threshold, res.misassign_rate, marker="o", label="misassignment rate")
ax.plot(res.threshold, res.hold_rate, marker="s", label="hold (미확정) rate")
ax.axhline(0.02, ls="--", c="r", lw=1, label="target 0.02")
ax.set_xlabel("cosine similarity threshold"); ax.set_ylabel("rate"); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f"Open-set identification, {len(uniq)} unseen cattle | GRAY={CONFIG['GRAY']}")
plt.tight_layout(); plt.savefig("/content/openset_curve.png", dpi=150); plt.show()

## 8. 한우 검증 — 이 셀이 "한우에 되냐"의 유일한 답이다

한우 사진 없이는 답이 없다. **재학습은 필요 없다.** 임베딩 방식이라 등록만 하면 된다.

필요한 최소량: **한우 20두 × 8장** (스마트폰, 30cm, 아랫입술 전체 포함).
농가 1회 방문 반나절이면 끝난다. 폴더 구조는 Zenodo와 동일하게 `한우/개체번호/사진.jpg`.

여기서 나온 숫자가 위 68두 결과보다 크게 낮으면 → 도메인 갭 확정 → 한우 사진으로 파인튜닝 추가.

In [ ]:
# 한우 이미지를 /content/hanwoo/<가축이력번호>/*.jpg 로 올린 뒤 실행
HANWOO = "/content/hanwoo"

if os.path.isdir(HANWOO):
    hp = []
    for e in exts: hp += glob.glob(os.path.join(HANWOO,"**",e), recursive=True)
    hdf = pd.DataFrame({"path":hp}); hdf["cow"]=hdf.path.apply(lambda p: os.path.basename(os.path.dirname(p)))
    print(f"한우 {hdf.cow.nunique()}두 / {len(hdf)}장")

    gp,gi,pp,pi = [],[],[],[]
    for cow,grp in hdf.groupby("cow"):
        ps = grp.path.sample(frac=1.0, random_state=SEED).tolist()
        if len(ps)<4: continue
        gp+=ps[:3]; gi+=[cow]*3; pp+=ps[3:]; pi+=[cow]*(len(ps)-3)
    Gh, Ph = embed(gp,CONFIG), embed(pp,CONFIG)
    uh = sorted(set(gi))
    ch = torch.stack([F.normalize(Gh[[i for i,c in enumerate(gi) if c==u]].mean(0),dim=0) for u in uh])
    s2 = Ph @ ch.T; bs2, bi2 = s2.max(1)
    pr = [uh[i] for i in bi2.tolist()]
    corr = np.array([a==b for a,b in zip(pr,pi)])
    print("한우 Top-1:", round(float(corr.mean()),4))
    for t in [0.4,0.5,0.6,0.7,0.8]:
        a = bs2.numpy()>=t
        print(f"  t={t}: 오배정률 {((a)&(~corr)).sum()/len(pi):.4f} / 보류율 {(~a).sum()/len(pi):.4f}")
else:
    print("한우 데이터 없음 — 이 셀은 사진을 모은 뒤 실행. 그 전까지 '한우 적용 가능'은 검증된 주장이 아니다.")

## 9. 배포용 내보내기 (ONNX) + 결과 백업

In [ ]:
enc.eval()
dummy = torch.randn(1,3,CONFIG["img_size"],CONFIG["img_size"], device=dev)
torch.onnx.export(enc, dummy, "/content/muzzle_encoder.onnx",
                  input_names=["image"], output_names=["embedding"],
                  dynamic_axes={"image":{0:"b"},"embedding":{0:"b"}}, opset_version=17)
print("ONNX 저장 완료")
!ls -lh /content/*.onnx /content/*.pt /content/*.csv /content/*.png

# 구글 드라이브에 백업 (세션 끊기면 /content는 전부 날아간다)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/muzzle
!cp /content/muzzle_encoder.pt /content/muzzle_encoder.onnx /content/openset_results.csv /content/openset_curve.png /content/drive/MyDrive/muzzle/
print("백업 완료")

## 10. 다음에 할 일 (순서대로)

1. `CONFIG["GRAY"]=False`, `exp_name="rgb"`로 바꾸고 **5~7번 셀 재실행** → `openset_results.csv`에 두 줄이 쌓인다. 이게 그레이스케일 실험 결과다.
2. 한우 사진 20두×8장 확보 → 8번 셀 실행
3. 등록 장수 3장 vs 5장 비교 (앱의 "몇 장 찍게 할 것인가"가 이 실험으로 결정된다)
4. `low_quality()`의 축소 비율을 고정값으로 바꿔가며 돌리면 **해상도 열화 실험**이 그대로 나온다

**인용**: Xiong, Li & Erickson (2022), Beef Cattle Muzzle/Noseprint database, Zenodo, DOI 10.5281/zenodo.6324361, CC BY 4.0. 발표자료에 반드시 표기할 것 (CC BY 조건).